# Pneumonia Detection from Chest X-ray Images — EDA

**Dataset:** Chest X-ray Balanced Dataset  
**Source:** https://www.kaggle.com/datasets/umitka/chest-x-ray-balanced  
**Task:** Binary classification — Normal vs Pneumonia  
**Structure:** train / val / test, each containing NORMAL and PNEUMONIA folders

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
import random

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
random.seed(42)
np.random.seed(42)

BASE_DIR = Path("chest_xray_balanced")
SPLITS = ["train", "val", "test"]
CLASSES = ["NORMAL", "PNEUMONIA"]

## 2. Load the Dataset

Build one master table listing every image file, its split (train/val/test), 
and its class (NORMAL/PNEUMONIA). This table is reused in every step below.

In [ ]:
def load_dataset_index(base_dir, splits, classes):
    records = []
    for split in splits:
        for cls in classes:
            folder = base_dir / split / cls
            if not folder.exists():
                print(f"WARNING: missing folder {folder}")
                continue
            for fname in os.listdir(folder):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    records.append({
                        "filename": fname,
                        "filepath": str(folder / fname),
                        "split": split,
                        "class": cls
                    })
    return pd.DataFrame(records)

df = load_dataset_index(BASE_DIR, SPLITS, CLASSES)
print(f"Total images loaded: {len(df)}")
df.head()

In [ ]:
# Quick counts
print(df.groupby(['split', 'class']).size())

## 3. Data Cleaning

Before analyzing anything, verify the data is trustworthy:
- Check for corrupted/unreadable images
- Check for duplicate files
- Check for suspiciously small/broken files

In [ ]:
# 3.1 Check for corrupted images
def check_corrupted(filepath):
    try:
        with Image.open(filepath) as img:
            img.verify()
        return True
    except Exception:
        return False

df['is_valid'] = df['filepath'].apply(check_corrupted)
print(f"Corrupted images found: {(~df['is_valid']).sum()}")

In [ ]:
# 3.2 Check for duplicates
duplicates = df[df.duplicated(subset=['filepath'], keep=False)]
print(f"Duplicate files found: {len(duplicates)}")

In [ ]:
# 3.3 Check for tiny/broken files
df['filesize_kb'] = df['filepath'].apply(lambda p: os.path.getsize(p) / 1024)
tiny_files = df[df['filesize_kb'] < 1]
print(f"Suspiciously small files: {len(tiny_files)}")
df['filesize_kb'].describe()

In [ ]:
# 3.4 Remove problem files, keep only the clean dataset
df_clean = df[df['is_valid']].drop_duplicates(subset=['filepath']).copy()
df_clean = df_clean[df_clean['filesize_kb'] >= 1]

print(f"Before cleaning: {len(df)}")
print(f"After cleaning:  {len(df_clean)}")

## 4. Exploration & Visualization

Now that the data is verified clean, explore it visually.

### 4.1 Class Distribution

In [ ]:
pivot_counts = df_clean.groupby(['split', 'class']).size().unstack().reindex(SPLITS)

fig, ax = plt.subplots(figsize=(10, 6))
pivot_counts.plot(kind="bar", ax=ax, color=["#4C72B0", "#C44E52"])
ax.set_title("Class Distribution Across Splits")
ax.set_ylabel("Number of Images")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("01_class_distribution.png", dpi=150)
plt.show()

### 4.2 Sample Images

In [ ]:
def show_samples(data, split="train", n_samples=4):
    fig, axes = plt.subplots(2, n_samples, figsize=(16, 8))
    for row, cls in enumerate(CLASSES):
        subset = data[(data['split'] == split) & (data['class'] == cls)]
        paths = subset['filepath'].sample(n_samples, random_state=1).tolist()
        for col, path in enumerate(paths):
            img = Image.open(path)
            axes[row, col].imshow(img, cmap="gray")
            axes[row, col].set_title(f"{cls}\n{img.size}", fontsize=10)
            axes[row, col].axis("off")
    plt.suptitle(f"Sample Images — {split.upper()}")
    plt.tight_layout()
    plt.savefig(f"02_sample_images_{split}.png", dpi=150)
    plt.show()

show_samples(df_clean, split="train")

### 4.3 Image Dimensions & Color Mode

In [ ]:
def inspect_images(data, sample_size=300):
    sample = data.sample(min(sample_size, len(data)), random_state=1)
    results = []
    for _, row in sample.iterrows():
        with Image.open(row['filepath']) as img:
            results.append({
                "class": row['class'], "width": img.size[0],
                "height": img.size[1], "mode": img.mode
            })
    return pd.DataFrame(results)

dim_df = inspect_images(df_clean)
print(dim_df[["width", "height"]].describe())
print(f"\nColor modes: {dim_df['mode'].value_counts().to_dict()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(dim_df["width"], dim_df["height"],
                c=dim_df["class"].map({"NORMAL": "#4C72B0", "PNEUMONIA": "#C44E52"}), alpha=0.5)
axes[0].set_title("Image Dimension Spread")
axes[0].set_xlabel("Width"); axes[0].set_ylabel("Height")

dim_df["mode"].value_counts().plot(kind="bar", ax=axes[1], color="#55A868")
axes[1].set_title("Color Mode Distribution")
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig("03_dimension_analysis.png", dpi=150)
plt.show()

### 4.4 Pixel Intensity Distribution

In [ ]:
def get_pixel_intensities(data, split="train", sample_size=100):
    intensities = {"NORMAL": [], "PNEUMONIA": []}
    for cls in CLASSES:
        subset = data[(data['split'] == split) & (data['class'] == cls)]
        paths = subset['filepath'].sample(min(sample_size, len(subset)), random_state=1).tolist()
        for path in paths:
            arr = np.array(Image.open(path).convert("L"))
            intensities[cls].extend(arr.flatten()[::50])
    return intensities

intensities = get_pixel_intensities(df_clean)

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(intensities["NORMAL"], bins=50, alpha=0.5, label="Normal", color="#4C72B0", density=True)
ax.hist(intensities["PNEUMONIA"], bins=50, alpha=0.5, label="Pneumonia", color="#C44E52", density=True)
ax.set_xlabel("Pixel Intensity (0-255)")
ax.set_title("Pixel Intensity: Normal vs Pneumonia")
ax.legend()
plt.tight_layout()
plt.savefig("04_pixel_intensity.png", dpi=150)
plt.show()

print(f"Normal    mean: {np.mean(intensities['NORMAL']):.2f}")
print(f"Pneumonia mean: {np.mean(intensities['PNEUMONIA']):.2f}")

### 4.5 Average Image per Class

Averaging many images per class can reveal systematic visual differences.

In [ ]:
def compute_mean_image(data, cls, split="train", img_size=(128,128), sample_size=200):
    subset = data[(data['split'] == split) & (data['class'] == cls)]
    paths = subset['filepath'].sample(min(sample_size, len(subset)), random_state=1).tolist()
    imgs = [np.array(Image.open(p).convert("L").resize(img_size), dtype=np.float32) for p in paths]
    return np.mean(imgs, axis=0)

mean_normal = compute_mean_image(df_clean, "NORMAL")
mean_pneumonia = compute_mean_image(df_clean, "PNEUMONIA")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(mean_normal, cmap="gray"); axes[0].set_title("Average Normal"); axes[0].axis("off")
axes[1].imshow(mean_pneumonia, cmap="gray"); axes[1].set_title("Average Pneumonia"); axes[1].axis("off")
axes[2].imshow(mean_pneumonia - mean_normal, cmap="bwr"); axes[2].set_title("Difference"); axes[2].axis("off")
plt.tight_layout()
plt.savefig("05_mean_image_comparison.png", dpi=150)
plt.show()

## 5. Preprocessing & Feature Engineering

Define the pipeline that will prepare raw images for model training in Week 4:
1. Resize to a fixed size
2. Convert to RGB (3 channels)
3. Normalize pixel values to [0,1]
4. Plan data augmentation

### 5.1 Resize & Normalize

In [ ]:
IMG_SIZE = (224, 224)  # compatible with DenseNet121 used later

def preprocess_image(filepath, target_size=IMG_SIZE):
    img = Image.open(filepath).convert("RGB")
    img = img.resize(target_size)
    return np.array(img) / 255.0

sample_path = df_clean[df_clean['class'] == 'PNEUMONIA']['filepath'].iloc[0]
processed = preprocess_image(sample_path)

print(f"Processed shape: {processed.shape}")
print(f"Pixel range: [{processed.min():.3f}, {processed.max():.3f}]")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(Image.open(sample_path), cmap="gray"); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(processed); axes[1].set_title("Resized + Normalized"); axes[1].axis("off")
plt.tight_layout()
plt.savefig("06_preprocessing_demo.png", dpi=150)
plt.show()

### 5.2 Data Augmentation Plan

Small rotations, shifts, zoom, and brightness changes — no horizontal flip, 
since X-ray anatomy should not be mirrored.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    brightness_range=[0.9, 1.1],
    horizontal_flip=False
)

sample_img = np.array(Image.open(sample_path).convert("RGB").resize(IMG_SIZE))
sample_img = sample_img.reshape((1,) + sample_img.shape)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
i = 0
for batch in datagen.flow(sample_img, batch_size=1):
    axes[i].imshow(batch[0].astype("uint8"))
    axes[i].axis("off")
    axes[i].set_title(f"Augmented {i+1}")
    i += 1
    if i == 5:
        break
plt.suptitle("Data Augmentation Preview")
plt.tight_layout()
plt.savefig("07_augmentation_preview.png", dpi=150)
plt.show()

### 5.3 Save Cleaned Index

Save the cleaned file list so Week 4 can load it directly without repeating cleaning.

In [ ]:
df_clean.to_csv("clean_dataset_index.csv", index=False)
print("Saved: clean_dataset_index.csv")

## 6. Summary of Findings

**Dataset**
- Total images: [X] → after cleaning: [X]
- Corrupted files removed: [X]
- Class balance confirmed across train/val/test

**Image Properties**
- Dimensions: [describe] → resizing to 224x224 needed
- Color mode: [grayscale/RGB] → converting to RGB for DenseNet121

**Observations**
- Pixel intensity: [describe difference between classes]
- Mean image comparison: [describe visual pattern difference]

**Preprocessing Plan for Week 4**
- Resize: 224x224
- Normalize: 0–1 scaling
- Augmentation: rotation, shift, zoom, brightness (no flip)